# Gold Layer - RetailMax Data Pipeline

Este notebook aplica reglas de negocio y crea el modelo dimensional final.

## 1. Instalar librerías y cargar configuración

In [ ]:
%pip install pandas python-dotenv azure-storage-file-datalake numpy -q

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO
from datetime import datetime, timedelta

# Cargar configuración
load_dotenv()

AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY = os.getenv("AZURE_STORAGE_KEY")

## 2. Conectar a ADLS y leer datos de Silver

In [ ]:
# Conectar a ADLS
service_client = DataLakeServiceClient(
    account_url=f"https://{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=AZURE_STORAGE_KEY
)
gold_fs = service_client.get_file_system_client("gold")
silver_fs = service_client.get_file_system_client("silver")

# Leer todos los archivos Parquet de Silver
dataframes = {}
for path in silver_fs.get_paths():
    if path.name.endswith(".parquet"):
        table_name = path.name.split("/")[1].replace(".parquet", "")
        file_client = silver_fs.get_file_client(path.name)
        df = pd.read_parquet(BytesIO(file_client.download_file().readall()))
        dataframes[table_name] = df
        print(f"✓ {table_name}: {len(df):,} registros")

## 3. Función auxiliar para subir a ADLS

In [ ]:
def upload_to_adls(df, table_name):
    """Sube dataframe a ADLS en formato Parquet"""
    temp_file = f"temp_{table_name}.parquet"
    df.to_parquet(temp_file, index=False)
    
    dir_client = gold_fs.get_directory_client(table_name.upper())
    try:
        dir_client.create_directory()
    except:
        pass
    
    file_client = dir_client.get_file_client(f"{table_name}.parquet")
    with open(temp_file, "rb") as data:
        file_client.upload_data(data, overwrite=True)
    
    os.remove(temp_file)
    print(f"✓ {table_name.upper()} cargado en Gold")

## 4. Transformaciones Gold

In [ ]:
# Asignar dataframes
dim_productos = dataframes["dim_productos"]
dim_tiendas = dataframes["dim_tiendas"]
dim_clientes = dataframes["dim_clientes"]
fact_ventas = dataframes["fact_ventas"]
fact_inventario = dataframes["fact_inventario"]
fact_devoluciones = dataframes["fact_devoluciones"]

# Enriquecer dimensiones con campos calculados
dim_productos['categoria_nivel_1'] = dim_productos['categoria'].apply(lambda x: str(x).split('_')[0] if '_' in str(x) else str(x))
dim_productos['margen_estimado'] = dim_productos['precio'] * 0.30

zona_map = {'CD_Bogota': 'ZONA_NORTE', 'CD_Ciudad_Mexico': 'ZONA_CENTRO', 'CD_Santiago': 'ZONA_SUR'}
dim_tiendas['zona_distribucion'] = dim_tiendas.get('centro_distribucion', pd.Series(['ZONA_PENDIENTE']*len(dim_tiendas))).map(zona_map).fillna('ZONA_PENDIENTE')

# Enriquecer hechos con dimensiones
fact_ventas = fact_ventas.merge(dim_productos[['id_articulo', 'categoria_nivel_1', 'margen_estimado']], on='id_articulo', how='left')
fact_ventas = fact_ventas.merge(dim_tiendas[['id_tienda', 'pais', 'zona_distribucion']], on='id_tienda', how='left')

fact_inventario = fact_inventario.merge(dim_productos[['id_articulo', 'categoria_nivel_1', 'nombre_producto']], on='id_articulo', how='left')
fact_inventario = fact_inventario.merge(dim_tiendas[['id_tienda', 'pais', 'ciudad']], on='id_tienda', how='left')

fact_devoluciones = fact_devoluciones.merge(dim_productos[['id_articulo', 'categoria_nivel_1', 'nombre_producto']], on='id_articulo', how='left')
fact_devoluciones = fact_devoluciones.merge(dim_tiendas[['id_tienda', 'pais', 'ciudad']], on='id_tienda', how='left')

print("✓ Transformaciones Gold completadas")

## 5. Cálculo RFM

In [ ]:
# Calcular RFM (Recencia, Frecuencia, Monto)
fact_ventas['fecha_hora'] = pd.to_datetime(fact_ventas['fecha_hora'], errors='coerce')
ventas_validas = fact_ventas[fact_ventas['fecha_hora'].notna()].copy()

fecha_ref = ventas_validas['fecha_hora'].max()
fecha_90 = fecha_ref - timedelta(days=90)
fecha_180 = fecha_ref - timedelta(days=180)

# Clientes activos (compra en últimos 180 días)
clientes_activos = ventas_validas[ventas_validas['fecha_hora'] >= fecha_180]['id_miembro'].unique()

# R: días desde última compra
recencia = ventas_validas[ventas_validas['id_miembro'].isin(clientes_activos)].groupby('id_miembro')['fecha_hora'].max().reset_index()
recencia['recencia_dias'] = (fecha_ref - recencia['fecha_hora']).dt.days

# F: número de transacciones en 90 días
frecuencia = ventas_validas[(ventas_validas['id_miembro'].isin(clientes_activos)) & (ventas_validas['fecha_hora'] >= fecha_90)].groupby('id_miembro').size().reset_index(name='frecuencia_90dias')

# M: monto total en 90 días
monto = ventas_validas[(ventas_validas['id_miembro'].isin(clientes_activos)) & (ventas_validas['fecha_hora'] >= fecha_90)].groupby('id_miembro')['vr_venta_neto'].sum().reset_index(name='monto_90dias')

# Combinar R, F, M
rfm = recencia.merge(frecuencia, on='id_miembro', how='outer').merge(monto, on='id_miembro', how='outer').fillna(0)

# Asignar scores 1-5 por quintiles
rfm['score_r'] = pd.qcut(rfm['recencia_dias'], 5, labels=[5,4,3,2,1], duplicates='drop').astype(int)
rfm['score_f'] = pd.qcut(rfm['frecuencia_90dias'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
rfm['score_m'] = pd.qcut(rfm['monto_90dias'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop').astype(int)

# Segmento RFM
rfm['segmento_rfm'] = rfm['score_r'].astype(str) + rfm['score_f'].astype(str) + rfm['score_m'].astype(str)
rfm['nombre_segmento'] = rfm.apply(lambda x: 'Champions' if x['score_r'] >= 4 and x['score_f'] >= 4 and x['score_m'] >= 4 else 'Others', axis=1)

fact_rfm_clientes = rfm
print(f"✓ RFM calculado: {len(fact_rfm_clientes):,} clientes")

## 6. Tablas de agregación (KPIs)

In [ ]:
# KPIs ejecutivos
kpi_ventas_pais_canal = fact_ventas.groupby(['pais', 'canal']).agg({'vr_venta_neto': 'sum', 'id_venta': 'count'}).reset_index()
kpi_ventas_pais_canal.columns = ['pais', 'canal', 'ventas_netas', 'num_transacciones']

kpi_top_articulos = fact_ventas.groupby(['categoria_nivel_1', 'id_articulo', 'nombre_producto']).agg({'vr_venta_neto': 'sum'}).reset_index()
kpi_top_articulos = kpi_top_articulos.groupby('categoria_nivel_1').apply(lambda x: x.nlargest(10, 'vr_venta_neto')).reset_index(drop=True)

# Tasa de devolución
unidades_vendidas = fact_ventas.groupby(['categoria_nivel_1', 'canal']).size().reset_index(name='vendidas')
unidades_devueltas = fact_devoluciones.groupby(['categoria_nivel_1', 'canal']).size().reset_index(name='devueltas')
tasa_devolucion = unidades_vendidas.merge(unidades_devueltas, on=['categoria_nivel_1', 'canal'], how='left').fillna(0)
tasa_devolucion['tasa_devolucion_pct'] = (tasa_devolucion['devueltas'] / tasa_devolucion['vendidas'] * 100).round(2)

print("✓ KPIs generados")

## 7. Documentación de linaje

In [ ]:
# Linaje de datos (simplificado)
linaje = pd.DataFrame([
    {'campo': 'vr_venta_neto', 'origen': 'fact_ventas', 'transformacion': 'precio - descuento', 'proposito': 'Valor neto de venta'},
    {'campo': 'cobertura_dias', 'origen': 'fact_inventario', 'transformacion': 'stock_fisico / consumo_promedio', 'proposito': 'Días de inventario disponible'},
    {'campo': 'segmento_rfm', 'origen': 'fact_ventas + dim_clientes', 'transformacion': 'Concatenación scores R-F-M', 'proposito': 'Segmentación clientes'}
])

## 8. Cargar datos a Gold

In [ ]:
# Cargar todas las tablas
upload_to_adls(dim_productos, 'dim_productos')
upload_to_adls(dim_tiendas, 'dim_tiendas')
upload_to_adls(dim_clientes, 'dim_clientes')
upload_to_adls(fact_ventas, 'fact_ventas')
upload_to_adls(fact_inventario, 'fact_inventario')
upload_to_adls(fact_devoluciones, 'fact_devoluciones')
upload_to_adls(fact_rfm_clientes, 'fact_rfm_clientes')
upload_to_adls(kpi_ventas_pais_canal, 'kpi_ventas_pais_canal')
upload_to_adls(kpi_top_articulos, 'kpi_top_articulos')
upload_to_adls(tasa_devolucion, 'kpi_tasa_devolucion')
upload_to_adls(linaje, 'linaje_datos')

## 9. Resumen

In [ ]:
print("=" * 50)
print("✓ Proceso Gold completado")
print(f"✓ 10 tablas cargadas en Gold")
print("=" * 50)

## Transformación Gold: fact_ventas (final)

**Regla de negocio:** Calcular vr_venta_neto = qty_vendida x precio_unitario - descuento; validar id_miembro contra dim_clientes o asignar cliente anónimo; agregar indicador de venta con descuento.

**Nota:** Estas transformaciones ya se aplicaron en Silver.

In [ ]:
fact_ventas_gold = fact_ventas.copy()

# Enriquecer con información de dimensiones para análisis
fact_ventas_gold = fact_ventas_gold.merge(
    dim_productos_gold[['id_articulo', 'categoria_nivel_1', 'margen_estimado']],
    on='id_articulo',
    how='left'
)

fact_ventas_gold = fact_ventas_gold.merge(
    dim_tiendas_gold[['id_tienda', 'pais', 'zona_distribucion']],
    on='id_tienda',
    how='left'
)

print(f"fact_ventas_gold: {len(fact_ventas_gold):,} registros")

## Transformación Gold: fact_inventario (final)

**Regla de negocio:** Calcular cobertura_dias = stock_fisico / promedio_consumo_14dias; flag alerta_quiebre cuando cobertura_dias sea menor a 7; calcular diferencia frente a stock_minimo_config.

**Nota:** Estas transformaciones ya se aplicaron en Silver.

In [ ]:
fact_inventario_gold = fact_inventario.copy()

# Enriquecer con información de productos
fact_inventario_gold = fact_inventario_gold.merge(
    dim_productos_gold[['id_articulo', 'categoria_nivel_1', 'nombre_producto']],
    on='id_articulo',
    how='left'
)

# Enriquecer con información de tiendas
fact_inventario_gold = fact_inventario_gold.merge(
    dim_tiendas_gold[['id_tienda', 'pais', 'ciudad']],
    on='id_tienda',
    how='left'
)

print(f"fact_inventario_gold: {len(fact_inventario_gold):,} registros")

## Transformación Gold: fact_devoluciones (final)

**Regla de negocio:** Join con la venta origen para obtener precio original; estandarizar motivo_cod a descripción legible; calcular tasa_devolucion por artículo y categoría.

**Nota:** Estas transformaciones ya se aplicaron en Silver.

In [ ]:
fact_devoluciones_gold = fact_devoluciones.copy()

# Enriquecer con información de productos
fact_devoluciones_gold = fact_devoluciones_gold.merge(
    dim_productos_gold[['id_articulo', 'categoria_nivel_1', 'nombre_producto']],
    on='id_articulo',
    how='left'
)

# Enriquecer con información de tiendas
fact_devoluciones_gold = fact_devoluciones_gold.merge(
    dim_tiendas_gold[['id_tienda', 'pais', 'ciudad']],
    on='id_tienda',
    how='left'
)

print(f"fact_devoluciones_gold: {len(fact_devoluciones_gold):,} registros")

## Transformación Gold: fact_rfm_clientes

**Regla de negocio:** Calcular R como días desde última transacción, F como número de transacciones en 90 días, M como valor monetario en 90 días; asignar score 1 a 5 por quintiles y construir segmento RFM.

**Regla de negocio específica:** El score RFM se calcula sobre los últimos 90 días. Cada dimensión recibe un puntaje de 1 a 5 usando quintiles sobre todos los clientes activos (al menos una compra en 180 días).

In [ ]:
# Filtrar ventas con fecha válida
fact_ventas_gold['fecha_hora'] = pd.to_datetime(fact_ventas_gold['fecha_hora'], errors='coerce')
ventas_validas = fact_ventas_gold[fact_ventas_gold['fecha_hora'].notna()].copy()

# Fecha de referencia para cálculos
fecha_referencia = ventas_validas['fecha_hora'].max()
fecha_90_dias = fecha_referencia - timedelta(days=90)
fecha_180_dias = fecha_referencia - timedelta(days=180)

print(f"Fecha de referencia: {fecha_referencia}")
print(f"Fecha corte 90 días: {fecha_90_dias}")
print(f"Fecha corte 180 días: {fecha_180_dias}")

In [ ]:
# Identificar clientes activos (compra en últimos 180 días)
clientes_activos = ventas_validas[ventas_validas['fecha_hora'] >= fecha_180_dias]['id_miembro'].unique()
print(f"Clientes activos (últimos 180 días): {len(clientes_activos):,}")

In [ ]:
# Calcular R (Recencia): días desde última transacción
recencia = ventas_validas[ventas_validas['id_miembro'].isin(clientes_activos)].groupby('id_miembro')['fecha_hora'].max().reset_index()
recencia['recencia_dias'] = (fecha_referencia - recencia['fecha_hora']).dt.days
recencia = recencia[['id_miembro', 'recencia_dias']]

# Calcular F (Frecuencia): número de transacciones en 90 días
frecuencia = ventas_validas[
    (ventas_validas['id_miembro'].isin(clientes_activos)) & 
    (ventas_validas['fecha_hora'] >= fecha_90_dias)
].groupby('id_miembro').size().reset_index(name='frecuencia_90dias')

# Calcular M (Monto): valor monetario en 90 días
monto = ventas_validas[
    (ventas_validas['id_miembro'].isin(clientes_activos)) & 
    (ventas_validas['fecha_hora'] >= fecha_90_dias)
].groupby('id_miembro')['vr_venta_neto'].sum().reset_index(name='monto_90dias')

# Combinar R, F, M
rfm_raw = recencia.merge(frecuencia, on='id_miembro', how='outer')
rfm_raw = rfm_raw.merge(monto, on='id_miembro', how='outer')
rfm_raw = rfm_raw.fillna(0)

print(f"RFM raw: {len(rfm_raw):,} clientes")

In [ ]:
# Asignar scores 1-5 usando quintiles
# R: menor recencia = mejor score (5)
rfm_raw['score_r'] = pd.qcut(rfm_raw['recencia_dias'], 5, labels=[5,4,3,2,1], duplicates='drop')

# F: mayor frecuencia = mejor score (5)
rfm_raw['score_f'] = pd.qcut(rfm_raw['frecuencia_90dias'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop')

# M: mayor monto = mejor score (5)
rfm_raw['score_m'] = pd.qcut(rfm_raw['monto_90dias'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop')

# Convertir scores a enteros
rfm_raw['score_r'] = rfm_raw['score_r'].astype(int)
rfm_raw['score_f'] = rfm_raw['score_f'].astype(int)
rfm_raw['score_m'] = rfm_raw['score_m'].astype(int)

# Construir segmento RFM concatenando scores
rfm_raw['segmento_rfm'] = rfm_raw['score_r'].astype(str) + rfm_raw['score_f'].astype(str) + rfm_raw['score_m'].astype(str)

# Asignar nombres de segmento basados en scores
def asignar_nombre_segmento(row):
    r, f, m = row['score_r'], row['score_f'], row['score_m']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 4 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and m >= 4:
        return 'Potential Loyalists'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    elif r <= 2:
        return 'At Risk'
    else:
        return 'Others'

rfm_raw['nombre_segmento'] = rfm_raw.apply(asignar_nombre_segmento, axis=1)

fact_rfm_clientes = rfm_raw

print(f"fact_rfm_clientes: {len(fact_rfm_clientes):,} clientes")
print(f"\nDistribución de segmentos:")
print(fact_rfm_clientes['nombre_segmento'].value_counts())

## Tabla de agregación: KPIs Ejecutivos

**Regla de negocio:** El dashboard ejecutivo consolida diariamente: ventas netas por país y canal, comparativo versus el mismo día de la semana anterior, top 10 de artículos por categoría y tasa de descuento promedio aplicada.

In [ ]:
# Ventas netas por país y canal
ventas_pais_canal = fact_ventas_gold.groupby(['pais', 'canal']).agg({
    'vr_venta_neto': 'sum',
    'id_venta': 'count'
}).reset_index()
ventas_pais_canal.columns = ['pais', 'canal', 'ventas_netas', 'num_transacciones']

# Tasa de descuento promedio aplicada
tasa_descuento = fact_ventas_gold[fact_ventas_gold['con_descuento'] == True].groupby(['pais', 'canal']).agg({
    'descuento': 'mean'
}).reset_index()
tasa_descuento.columns = ['pais', 'canal', 'tasa_descuento_promedio']

# Top 10 artículos por categoría
top_articulos_categoria = fact_ventas_gold.groupby(['categoria_nivel_1', 'id_articulo', 'nombre_producto']).agg({
    'vr_venta_neto': 'sum',
    'id_venta': 'count'
}).reset_index()
top_articulos_categoria.columns = ['categoria', 'id_articulo', 'nombre_producto', 'ventas_netas', 'num_transacciones']

# Obtener top 10 por categoría
top_10_por_categoria = top_articulos_categoria.groupby('categoria').apply(
    lambda x: x.nlargest(10, 'ventas_netas')
).reset_index(drop=True)

kpi_ejecutivos = {
    'ventas_pais_canal': ventas_pais_canal,
    'tasa_descuento_promedio': tasa_descuento,
    'top_10_articulos_categoria': top_10_por_categoria
}

print("KPIs ejecutivos generados:")
print(f"- Ventas por país/canal: {len(ventas_pais_canal):,} registros")
print(f"- Tasa descuento: {len(tasa_descuento):,} registros")
print(f"- Top 10 artículos: {len(top_10_por_categoria):,} registros")

## Tabla de agregación: Tasa de devolución

**Regla de negocio:** La tasa de devolución por categoría se calcula como unidades devueltas sobre unidades vendidas en el mismo periodo, expresada en porcentaje, por categoría de nivel 1 y por canal de venta.

In [ ]:
# Unidades vendidas por categoría y canal
unidades_vendidas = fact_ventas_gold.groupby(['categoria_nivel_1', 'canal']).agg({
    'id_venta': 'count'
}).reset_index()
unidades_vendidas.columns = ['categoria', 'canal', 'unidades_vendidas']

# Unidades devueltas por categoría y canal
unidades_devueltas = fact_devoluciones_gold.groupby(['categoria_nivel_1', 'canal']).agg({
    'id_venta': 'count'
}).reset_index()
unidades_devueltas.columns = ['categoria', 'canal', 'unidades_devueltas']

# Calcular tasa de devolución
tasa_devolucion = unidades_vendidas.merge(unidades_devueltas, on=['categoria', 'canal'], how='left')
tasa_devolucion['unidades_devueltas'] = tasa_devolucion['unidades_devueltas'].fillna(0)
tasa_devolucion['tasa_devolucion_pct'] = (tasa_devolucion['unidades_devueltas'] / tasa_devolucion['unidades_vendidas'] * 100).round(2)

print(f"tasa_devolucion: {len(tasa_devolucion):,} registros")

## Documentación de linaje de datos

**Requisito:** Documentar el linaje de al menos tres campos calculados en la capa Gold: tabla de origen, transformaciones aplicadas en orden, propósito del campo.

In [ ]:
linaje_datos = [
    {
        'campo_calculado': 'vr_venta_neto',
        'tabla_origen': 'fact_ventas (Silver)',
        'transformaciones': '1. Resta de descuento al precio unitario (precio - descuento) en Silver. 2. Join con dim_productos para enriquecer con categoría en Gold.',
        'proposito': 'Valor neto de la venta después de descuentos, usado para cálculos de ingresos y análisis de rentabilidad.'
    },
    {
        'campo_calculado': 'cobertura_dias',
        'tabla_origen': 'inv_stock_diario (Bronze) -> fact_inventario (Silver)',
        'transformaciones': '1. Cálculo de stock_fisico / (promedio_consumo_14dias + 1) en Silver. 2. Flag alerta_quiebre cuando cobertura_dias < 7 en Silver. 3. Join con dimensiones en Gold.',
        'proposito': 'Días de inventario disponible basado en velocidad de consumo. Usado para alertas de quiebre de stock y planificación de reabastecimiento.'
    },
    {
        'campo_calculado': 'segmento_rfm',
        'tabla_origen': 'fact_ventas (Silver) + CRM_MIEMBROS (Bronze)',
        'transformaciones': '1. Cálculo de R (recencia días), F (frecuencia 90 días), M (monto 90 días) en Gold. 2. Asignación de scores 1-5 por quintiles. 3. Concatenación de scores para formar segmento (ej: R5-F4-M5).',
        'proposito': 'Segmentación de clientes para campañas de marketing personalizadas. Permite identificar Champions, Loyal Customers, At Risk, etc.'
    }
]

df_linaje = pd.DataFrame(linaje_datos)
print("Linaje de datos - Campos calculados en Gold:")
print(df_linaje.to_string(index=False))

## Carga de datos transformados a ADLS Gold

In [ ]:
# Cargar dimensiones
upload_to_adls(dim_productos_gold, gold_fs, 'dim_productos')
upload_to_adls(dim_tiendas_gold, gold_fs, 'dim_tiendas')
upload_to_adls(dim_clientes_gold, gold_fs, 'dim_clientes')

# Cargar hechos
upload_to_adls(fact_ventas_gold, gold_fs, 'fact_ventas')
upload_to_adls(fact_inventario_gold, gold_fs, 'fact_inventario')
upload_to_adls(fact_devoluciones_gold, gold_fs, 'fact_devoluciones')
upload_to_adls(fact_rfm_clientes, gold_fs, 'fact_rfm_clientes')

# Cargar tablas de agregación
upload_to_adls(kpi_ejecutivos['ventas_pais_canal'], gold_fs, 'kpi_ventas_pais_canal')
upload_to_adls(kpi_ejecutivos['tasa_descuento_promedio'], gold_fs, 'kpi_tasa_descuento')
upload_to_adls(kpi_ejecutivos['top_10_articulos_categoria'], gold_fs, 'kpi_top_10_articulos')
upload_to_adls(tasa_devolucion, gold_fs, 'kpi_tasa_devolucion')

# Cargar documentación de linaje
upload_to_adls(df_linaje, gold_fs, 'linaje_datos')

## Resumen de ejecución

In [ ]:
print("=" * 60)
print("Proceso Gold finalizado exitosamente.")
print("\nTablas dimensiones cargadas:")
print("- dim_productos")
print("- dim_tiendas")
print("- dim_clientes")
print("\nTablas hechos cargadas:")
print("- fact_ventas")
print("- fact_inventario")
print("- fact_devoluciones")
print("- fact_rfm_clientes")
print("\nTablas de agregación/KPIs cargadas:")
print("- kpi_ventas_pais_canal")
print("- kpi_tasa_descuento")
print("- kpi_top_10_articulos")
print("- kpi_tasa_devolucion")
print("\nDocumentación:")
print("- linaje_datos")
print("=" * 60)